# 第 03 课 - 智能体设计模式

本课我们探索构建高效 AI 智能体的三种基础设计模式：

1. **清晰的智能体指令**——编写精确、定义角色的提示词来引导智能体的行为
2. **用 Pydantic 模型实现结构化输出**——确保智能体返回可预测、经过校验的数据
3. **单一职责智能体**——设计专注的智能体，每个只把一件事做好

我们会把每种模式应用到一个**旅行目的地推荐**场景中，逐步构建出一个能推荐目的地、检查可预订情况并处理行程安排的系统。

## 环境准备

前置条件：在仓库根目录运行 `pip install -r requirements.txt`，把 `.env.example` 复制为 `.env`，填入 `LLM_BASE_URL`、`LLM_API_KEY`、`LLM_MODEL`，然后运行 `python scripts/check_endpoint.py`。

下面的单元会从 `.env` 加载这些变量并构建聊天模型客户端。`ChatOpenAI` 使用 OpenAI Chat Completions 协议，所以同一段代码可以对接 DeepSeek、OpenAI、本地 Ollama 服务或任何其他兼容接口——只需要改这三个环境变量。`LLM_EXTRA_BODY` 用来传递可选的、特定于服务商的请求选项（对 DeepSeek 来说，它会关闭思考模式）。本 notebook 里的每一个智能体都由这同一个 `llm` 客户端构建。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"缺少必需的环境变量：{'、'.join(missing)}。"
        "请把仓库根目录下的 .env.example 复制为 .env 并填好。"
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"模型客户端已就绪：{os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

模型客户端已就绪：deepseek-v4-pro @ https://api.deepseek.com/v1


## 模式 1：清晰的智能体指令

影响最大的模式也是最简单的：为你的智能体编写清晰、详细的指令。

好的指令会定义：
- 智能体**是谁**（角色与语气）
- 它**该做什么**（分步骤的职责）
- 它**该如何表现**（约束与风格）

下面我们创建一个旅行礼宾智能体，它带有明确的指令，这些指令会塑造它产生的每一条回复。使用 `create_agent` 时，指令通过 `system_prompt` 传入；这个智能体没有工具，所以每条回复都直接来自模型，由该提示词引导。因此它的指令要求它坦率说明：实时的可预订情况和价格需要依赖当前的预订数据源。

`agent.invoke` 返回完整的消息历史——用户消息、任何工具调用，以及最终回复。`reply_text` 从最后一条消息里取出文本（有些服务商返回的 content 是一组内容块的列表，而不是单个字符串）；我们在这里定义一次，然后在本 notebook 的每个智能体上复用。

注意观察输出：礼宾智能体可能会先询问你的偏好，而不是直接给出推荐——这是它在执行指令中的第 1 步，并不是失败。

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[],
    system_prompt="""你是一位名叫小安的高端旅行礼宾顾问。你的职责是：
1. 了解旅行者的偏好（预算、气候、想玩什么）
2. 推荐符合这些偏好的目的地
3. 说明实时的可预订情况和价格必须通过当前的预订渠道核实
4. 每次都要提到签证要求和最佳旅行季节
语气热情、专业，对旅行充满热忱。请用中文回答。""",
)


def reply_text(result) -> str:
    """返回最后一条消息的文本；content 可能是字符串，也可能是内容块列表。"""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "我想在国内找个美食和历史都很棒的地方度假一周，预算 1.5 万元左右。",
            }
        ]
    }
)
print(reply_text(result))

很高兴为您规划这趟美食与历史兼具的假期！在国内一万元左右的预算，玩一周并且兼顾深度体验，是很充裕的。请问您更偏向南方还是北方？另外，您对气候有什么要求吗？比如想去暖和一点的地方，还是不怕冷、可以接受冬春季节的江南？


## 模式 2：用 Pydantic 模型实现结构化输出

自由文本适合对话，但下游系统需要结构化数据。
把 **Pydantic 模型**和**工具函数**配合使用，我们可以：

- 为智能体的输出定义精确的 schema
- 自动校验回复
- 可靠地把智能体的结果集成到应用逻辑中

强制执行的关键在于创建智能体时传入 `response_format=TravelRecommendations`。LangChain 会根据模型名称，用两种方式之一来强制执行该 schema：对于它已知支持原生结构化输出的模型（例如 `gpt-4o`），它使用服务商的 `response_format`；对于其他模型（包括这里使用的 DeepSeek 模型），它把 `TravelRecommendations` 注册为一个额外的工具，模型的最后一个动作就是调用这个工具，因此调用参数*就是*经过校验的对象，不需要模型再做任何 JSON 解析。无论哪种方式，实例都会出现在 `result["structured_response"]` 里，而不是以自由文本形式返回。`get_destination_details` 工具同样返回一个带类型的 `DestinationRecommendation`（LangChain 会把实例渲染成文本交给模型），所以对话的两端数据都经过了校验。

In [3]:
from typing import Annotated

from langchain.tools import tool
from pydantic import BaseModel, Field


class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_rmb: int = Field(description="一周行程的预估总预算，单位：人民币元")


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool
def get_destination_details(
    destination: Annotated[str, "要查询的目的地"]
) -> DestinationRecommendation:
    """获取某个度假目的地的结构化详情。

    覆盖成都、西安和哈尔滨。其他目的地一律返回 available=False 且没有数据，
    所以请从覆盖的目的地中挑选。
    """
    details = {
        "成都": DestinationRecommendation(
            destination="成都",
            available=True,
            best_season="3-5月",
            highlights=["大熊猫", "茶馆", "夜生活"],
            estimated_budget_rmb=14000,
        ),
        "西安": DestinationRecommendation(
            destination="西安",
            available=True,
            best_season="9-10月",
            highlights=["历史文化", "美食", "古迹"],
            estimated_budget_rmb=18000,
        ),
        "哈尔滨": DestinationRecommendation(
            destination="哈尔滨",
            available=False,
            best_season="12-2月",
            highlights=["冰雪", "自然风光", "探险"],
            estimated_budget_rmb=13000,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="未知",
            highlights=[],
            estimated_budget_rmb=0,
        ),
    )


# 传入 `response_format` 后，智能体返回的是一个经过校验的
# TravelRecommendations 对象，而不是自由文本。
structured_agent = create_agent(
    llm,
    tools=[get_destination_details],
    system_prompt="你是一名旅行专家，根据旅行者的偏好推荐目的地。请使用 get_destination_details 工具。请用中文回答。",
    response_format=TravelRecommendations,
)

result = structured_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "给一位热爱文化、预算 1.8 万元的旅行者推荐 3 个目的地"}
        ]
    }
)

plan: TravelRecommendations = result["structured_response"]
for rec in plan.recommendations:
    status = "可预订" if rec.available else "暂不可预订"
    print(f"{rec.destination}（{status}）")
    print(f"  最佳季节：{rec.best_season}")
    print(f"  亮点：{'、'.join(rec.highlights)}")
    print(f"  预估预算：{rec.estimated_budget_rmb} 元")
    print()
print(f"备注：{plan.personalized_note}")

西安（可预订）
  最佳季节：9-10月
  亮点：历史文化、美食、古迹
  预估预算：18000 元

成都（可预订）
  最佳季节：3-5月
  亮点：大熊猫、茶馆、夜生活
  预估预算：14000 元

哈尔滨（暂不可预订）
  最佳季节：12-2月
  亮点：冰雪、自然风光、探险
  预估预算：13000 元

备注：这位旅行者热爱文化且预算 1.8 万元。西安以历史文化、古迹和美食见长，一周预算约 1.8 万元，正好贴合预算，是最契合文化诉求的首选；成都以茶馆文化、夜生活和大熊猫为亮点，预算约 1.4 万元，富余可升级体验；哈尔滨目前无可用详情，暂不推荐。


## 模式 3：单一职责智能体

复杂任务受益于把工作拆分给多个专注的智能体，每个智能体只承担一项职责：

- 一个**目的地专家**，负责评估可预订情况并恰好选出一个目的地
- 一个**行程规划师**，负责处理航班、酒店和行程安排

这与软件工程中的*关注点分离*原则相呼应——每个智能体都更容易独立地测试、维护和改进。

下面，两个智能体依次运行。目的地专家返回一个经过校验的 `DestinationChoice`；它的 schema 把选择限制在工具报告为可预订的目的地之内。这个唯一的选择被传给行程规划师，这样规划师就能专注处理行程安排，而不必做目的地决策。第 08 课会展示如何把这样的交接接入 LangGraph 工作流，而不是手工完成。

In [4]:
from typing import Literal


class DestinationChoice(BaseModel):
    destination: Literal["成都", "西安"]
    best_season: str
    estimated_budget_rmb: int = Field(description="预估预算，单位：人民币元")
    pros: list[str]
    cons: list[str]
    rationale: str


destination_agent = create_agent(
    llm,
    tools=[get_destination_details],
    system_prompt="""你是一名目的地研究专家。你唯一的工作是：
1. 根据旅行者的偏好评估目的地
2. 用提供的工具查询是否可预订
3. 恰好选出一个可预订的目的地，并说明它的优缺点
请用中文回答。
不要讨论机票、酒店或行程安排——那由另一个智能体负责。""",
    response_format=DestinationChoice,
)

logistics_agent = create_agent(
    llm,
    tools=[],
    system_prompt="""你是一名旅行行程规划师。你唯一的工作是：
1. 为选定的目的地制定逐日行程
2. 在给定预算内给出机票和酒店方案
3. 说明签证要求取决于旅行者的护照，如果不知道就询问
4. 把机票和酒店价格标注为估算，绝不暗示实时有余位
5. 绝不假设旅行者的出发地、国籍或护照
请用中文回答。
不要推荐或更换目的地——那由另一个智能体负责。""",
)

# 第 1 步：目的地专家选出一个可预订的目的地
dest_result = destination_agent.invoke(
    {"messages": [{"role": "user", "content": "我想花一周时间体验文化和美食，预算不超过 1.8 万元。该去哪儿？"}]}
)
destination_choice: DestinationChoice = dest_result["structured_response"]
print("=== 目的地专家 ===")
print(destination_choice.model_dump_json(indent=2))

# 第 2 步：行程规划师制定出行计划
logistics_result = logistics_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "为这个已选定的目的地规划一周的行程，"
                    "不要换成其他目的地：\n"
                    f"{destination_choice.model_dump_json()}"
                ),
            }
        ]
    }
)
print("\n=== 行程规划师 ===")
print(reply_text(logistics_result))

=== 目的地专家 ===
{
  "destination": "西安",
  "best_season": "9-10月",
  "estimated_budget_rmb": 18000,
  "pros": [
    "历史文化底蕴深厚，兵马俑、古城墙等古迹众多",
    "美食种类丰富，以面食和西北风味为主",
    "一周时间适合深度游览古迹与品尝美食"
  ],
  "cons": [
    "预算处于上限（约1.8万）",
    "最佳季节为9-10月，其他季节体验略打折扣"
  ],
  "rationale": "旅行者希望一周体验文化和美食，预算不超过1.8万。西安文化古迹资源极其丰富，且以美食闻名，非常契合两大主题；一周时间足够深度游玩，预算刚好在上限内且可预订。成都虽美食与文化俱佳且预算更低，但西安的历史厚度与\"文化+美食\"双主题契合度更高。哈尔滨当前不可预订，故排除。"
}

=== 行程规划师 ===
### 西安一周行程规划（文化与美食主题）

#### 行程总览
- **天数**：7天6晚
- **预算**：约18,000元人民币（上限，尽量留出余量）
- **适合季节**：9-10月最佳；其他季节也可出行但体验略有差异

---

### 第1天：抵达西安，初探回民街
- **上午/下午**：抵达西安咸阳国际机场，入住酒店（建议住钟楼/南门附近，交通便利）
- **傍晚**：游览**钟楼、鼓楼**外观，漫步**回民街**，品尝羊肉泡馍、肉夹馍、凉皮等
- **晚餐**：回民街小吃或老孙家泡馍

---

### 第2天：兵马俑 + 华清宫
- **上午**：前往**秦始皇兵马俑博物馆**（建议请讲解或租讲解器）
- **中午**：景区附近简餐
- **下午**：游览**华清宫**（骊山、温泉遗址）
- **晚上**：返回市区，晚餐推荐**biangbiang面**或葫芦头

---

### 第3天：古城墙 + 碑林 + 书院门
- **上午**：骑行或步行**西安城墙**（永宁门上，含光门下）
- **中午**：城墙附近午餐
- **下午**：参观**碑林博物馆**，逛**书院门文化街**
- **晚上**：观看**《梦长安—大唐迎宾盛礼》**（如演出开放）或自由活动

---

### 第4天：陕西历史博物馆 +

## 小结

本课我们把三种智能体设计模式应用到了旅行推荐场景中：

| 模式 | 核心思想 | 收益 |
|---|---|---|
| **清晰的指令** | 在 `system_prompt` 里定义角色、职责和约束 | 一致、符合定位的智能体行为 |
| **结构化输出** | 把 Pydantic 模型作为 `response_format=` 传入，并读取 `result["structured_response"]` | 经过校验、机器可读的结果 |
| **单一职责** | 让每个智能体只负责一项专注的工作 | 更容易测试、维护和组合 |

这些模式可以自然地组合——你可以在一个单一职责智能体里同时使用清晰的指令和结构化输出，构建出健壮、可用于生产的系统。